# 🚀 Z-Image Turbo LoRA Training - Complete Production System

**Full-featured training with:**
- ✅ Complete W&B integration (auto metrics, images, artifacts)
- ✅ Dataset preprocessing (flip, resize, validate)
- ✅ Experiment ablations
- ✅ X/Y comparison grids
- ✅ Auto backup (Drive + HuggingFace)
- ✅ Quality metrics tracking

---

## 📦 Step 1: Installation

In [ ]:
%%capture
# Fix NumPy compatibility
!pip uninstall numpy scipy -y -q
!pip install "numpy<2.0" "scipy<1.14" -q

# Install dependencies
!pip install wandb pillow matplotlib tqdm huggingface_hub gdown torch diffusers transformers accelerate -q

# Clone AI Toolkit
import os
if not os.path.exists('/content/ai-toolkit'):
    !git clone https://github.com/ostris/ai-toolkit.git /content/ai-toolkit
    !cd /content/ai-toolkit && git submodule update --init --recursive
    !pip install -e /content/ai-toolkit -q

print("✓ Installation complete! Please RESTART RUNTIME now.")
print("  (Runtime > Restart runtime)")

## 🔄 After Restart: Install Helper Modules

In [ ]:
# ============================================================================
# INSTALL HELPER MODULES FROM GITHUB
# ============================================================================
# Update this with your GitHub username after creating the repo
GITHUB_REPO = "YOUR_USERNAME/zimage-turbo-training"  # ⚠ UPDATE THIS!

# Clone the repo
import os
if not os.path.exists('/content/zimage-helpers'):
    !git clone https://github.com/{GITHUB_REPO}.git /content/zimage-helpers
    print("✓ Helper modules cloned")
else:
    print("✓ Helper modules already present")

# Add to Python path
import sys
if '/content/zimage-helpers' not in sys.path:
    sys.path.insert(0, '/content/zimage-helpers')

# Verify imports work
try:
    from wandb_monitor import TrainingMonitor
    from dataset_processor import DatasetProcessor
    from xy_grid import XYGridGenerator
    from upload_utils import UploadManager
    print("\n✓ All helper modules imported successfully!")
except ImportError as e:
    print(f"\n❌ Import error: {e}")
    print("\nMake sure you:")
    print("  1. Created a GitHub repo")
    print("  2. Uploaded all .py files to it")
    print("  3. Updated GITHUB_REPO variable above")

## 🔑 Step 2: Authentication

In [ ]:
import os
import wandb
from huggingface_hub import login
import getpass

# ===== HUGGINGFACE =====
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except:
    HF_TOKEN = os.environ.get('HF_TOKEN')

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("HuggingFace Token: ")

if HF_TOKEN:
    login(token=HF_TOKEN)
    os.environ['HF_TOKEN'] = HF_TOKEN
    os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
    print("✓ HuggingFace authenticated")
else:
    print("⚠ HuggingFace skipped")

# ===== WANDB =====
try:
    WANDB_KEY = userdata.get('WANDB_API_KEY')
except:
    WANDB_KEY = os.environ.get('WANDB_API_KEY')

WANDB_ENABLED = False
use_wandb = input("Use W&B for tracking? (y/n): ").strip().lower()

if use_wandb == 'y':
    if not WANDB_KEY:
        WANDB_KEY = getpass.getpass("W&B API Key: ")
    
    if WANDB_KEY:
        wandb.login(key=WANDB_KEY, relogin=True)
        os.environ['WANDB_API_KEY'] = WANDB_KEY
        WANDB_ENABLED = True
        print("✓ W&B authenticated")
else:
    os.environ['WANDB_DISABLED'] = 'true'
    print("⚠ W&B disabled")

# ===== GOOGLE DRIVE =====
DRIVE_ENABLED = False
use_drive = input("Mount Google Drive? (y/n): ").strip().lower()

if use_drive == 'y':
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ENABLED = True
    print("✓ Google Drive mounted")
else:
    print("⚠ Google Drive skipped")

print("\n" + "="*60)
print("SETUP COMPLETE")
print("="*60)

## ⚙️ Step 3: Training Configuration

In [ ]:
from collections import OrderedDict
import random

# ========== PROJECT ==========
PROJECT_NAME = "my_zimage_lora"
RUN_NAME = None  # Auto-generate if None
GLOBAL_SEED = 42

# ========== WANDB ==========
WANDB_PROJECT = "zimage-turbo-training"
WANDB_ENTITY = None  # Your W&B username/team

# ========== MODEL ==========
MODEL_CONFIG = {
    'name_or_path': 'Tongyi-MAI/Z-Image-Turbo',
    'arch': 'zimage',
    'assistant_lora_path': 'ostris/zimage_turbo_training_adapter/zimage_turbo_training_adapter_v1.safetensors',  # or v2
    'quantize': True,
    'qtype': 'qfloat8',  # qfloat8 or None
    'quantize_te': True,
    'qtype_te': 'qfloat8',
    'low_vram': True,
}

# ========== NETWORK ==========
NETWORK_CONFIG = {
    'type': 'lora',  # 'lora' or 'dora'
    'linear': 16,  # LoRA rank
    'linear_alpha': 16,
}

# ========== TRAINING ==========
TRAINING_CONFIG = {
    'steps': 2000,
    'batch_size': 1,
    'learning_rate': 1e-4,
    'optimizer': 'adamw8bit',
    'gradient_accumulation_steps': 1,
    'gradient_checkpointing': True,
    'save_every': 250,
    'max_step_saves_to_keep': 4,
}

# ========== SAMPLING ==========
SAMPLE_CONFIG = {
    'sample_every': 200,
    'sample_steps': 8,
    'guidance_scale': 0,
    'seed': GLOBAL_SEED,
    'walk_seed': True,
}

# Sample prompts with dimensions
SAMPLE_PROMPTS = [
    {"prompt": "a beautiful landscape", "width": 1024, "height": 1024},
    {"prompt": "portrait of a woman", "width": 1024, "height": 1024},
    {"prompt": "a cute cat", "width": 768, "height": 768},
]

# Set seeds
random.seed(GLOBAL_SEED)
import numpy as np
np.random.seed(GLOBAL_SEED)

print("✓ Configuration loaded")
print(f"  Project: {PROJECT_NAME}")
print(f"  Network: {NETWORK_CONFIG['type'].upper()} (rank={NETWORK_CONFIG['linear']})")
print(f"  Steps: {TRAINING_CONFIG['steps']} @ LR={TRAINING_CONFIG['learning_rate']}")
print(f"  Adapter: {'v2' if 'v2' in MODEL_CONFIG['assistant_lora_path'] else 'v1'}")

## 📁 Step 4: Dataset Preparation

In [ ]:
from pathlib import Path

# Create upload directory
RAW_DATASET = Path('/content/raw_dataset')
RAW_DATASET.mkdir(exist_ok=True)

print("📁 UPLOAD YOUR IMAGES HERE:")
print(f"   {RAW_DATASET}")
print("\nFormat:")
print("  - image1.png / image1.jpg / image1.webp")
print("  - image1.txt (caption file, optional)")
print("\nThen run the next cell to process.")

In [ ]:
# Import processor
import sys
sys.path.append('/content')
from dataset_processor import DatasetProcessor, create_dataset_config

# Configure datasets
DATASET_CONFIGS = [
    create_dataset_config(
        name='main',
        source_folder='/content/raw_dataset',
        flip_horizontal=False,  # Create horizontal flips
        flip_vertical=False,    # Create vertical flips
        repeats=1,              # Repeat dataset N times
        resolutions=[512, 768, 1024],
    ),
]

# Process
processor = DatasetProcessor(output_dir='/content/dataset')
stats = processor.process_multiple_datasets(DATASET_CONFIGS)

# Validate
validation = processor.validate_dataset()

# Save stats for W&B
num_images = validation['total_images']
num_captions = validation['total_captions']

## 🔬 Step 5: Experiment Configuration (Optional)

In [ ]:
# ========== EXPERIMENT MODE ==========
EXPERIMENT_MODE = False  # Set True for ablations

EXPERIMENT_VARIATIONS = [
    {'name': 'baseline', 'learning_rate': 1e-4, 'rank': 16},
    {'name': 'high_lr', 'learning_rate': 2e-4, 'rank': 16},
    {'name': 'low_lr', 'learning_rate': 5e-5, 'rank': 16},
]

if EXPERIMENT_MODE:
    print(f"🔬 EXPERIMENT MODE: {len(EXPERIMENT_VARIATIONS)} variations")
    for v in EXPERIMENT_VARIATIONS:
        print(f"   - {v['name']}: LR={v['learning_rate']}, Rank={v['rank']}")
else:
    print("✓ Single run mode")

## 🎯 Step 6: Training with W&B

In [ ]:
import sys
sys.path.append('/content/ai-toolkit')
sys.path.append('/content')

from toolkit.job import run_job
from wandb_monitor import TrainingMonitor
import time
from datetime import datetime

def create_job_config(variation=None):
    """Create training job configuration"""
    
    # Apply variation if provided
    config = TRAINING_CONFIG.copy()
    network = NETWORK_CONFIG.copy()
    
    if variation:
        config['learning_rate'] = variation.get('learning_rate', config['learning_rate'])
        network['linear'] = variation.get('rank', network['linear'])
        network['linear_alpha'] = network['linear']
        name = f"{PROJECT_NAME}_{variation['name']}"
    else:
        name = RUN_NAME or PROJECT_NAME
    
    return OrderedDict([
        ('job', 'extension'),
        ('config', OrderedDict([
            ('name', name),
            ('process', [
                OrderedDict([
                    ('type', 'sd_trainer'),
                    ('training_folder', '/content/output'),
                    ('device', 'cuda:0'),
                    ('network', network),
                    ('save', OrderedDict([
                        ('dtype', 'float16'),
                        ('save_every', config['save_every']),
                        ('max_step_saves_to_keep', config['max_step_saves_to_keep'])
                    ])),
                    ('datasets', [
                        OrderedDict([
                            ('folder_path', '/content/dataset'),
                            ('caption_ext', 'txt'),
                            ('caption_dropout_rate', 0.05),
                            ('shuffle_tokens', False),
                            ('cache_latents_to_disk', True),
                            ('resolution', DATASET_CONFIGS[0]['resolutions'])
                        ])
                    ]),
                    ('train', OrderedDict([
                        ('batch_size', config['batch_size']),
                        ('steps', config['steps']),
                        ('gradient_accumulation_steps', config['gradient_accumulation_steps']),
                        ('train_unet', True),
                        ('train_text_encoder', False),
                        ('gradient_checkpointing', config['gradient_checkpointing']),
                        ('noise_scheduler', 'flowmatch'),
                        ('optimizer', config['optimizer']),
                        ('lr', config['learning_rate']),
                        ('cache_text_embeddings', True),
                        ('dtype', 'bf16'),
                    ])),
                    ('model', MODEL_CONFIG),
                    ('sample', OrderedDict([
                        ('sampler', 'flowmatch'),
                        ('sample_every', SAMPLE_CONFIG['sample_every']),
                        ('prompts', [p['prompt'] for p in SAMPLE_PROMPTS]),
                        ('width', SAMPLE_PROMPTS[0]['width']),
                        ('height', SAMPLE_PROMPTS[0]['height']),
                        ('neg', ''),
                        ('seed', SAMPLE_CONFIG['seed']),
                        ('walk_seed', SAMPLE_CONFIG['walk_seed']),
                        ('guidance_scale', SAMPLE_CONFIG['guidance_scale']),
                        ('sample_steps', SAMPLE_CONFIG['sample_steps'])
                    ]))
                ])
            ])
        ])),
        ('meta', OrderedDict([
            ('name', f'[{name}]'),
            ('version', '1.0')
        ]))
    ]), config, network

def run_training_with_wandb(variation=None):
    """Run training with W&B monitoring"""
    
    job_config, train_config, network_config = create_job_config(variation)
    
    run_name = job_config['config']['name']
    group_name = PROJECT_NAME if variation else None
    
    # Initialize W&B monitor if enabled
    if WANDB_ENABLED:
        config_dict = {
            'project_name': PROJECT_NAME,
            'model_path': MODEL_CONFIG['name_or_path'],
            'adapter_version': 'v2' if 'v2' in MODEL_CONFIG['assistant_lora_path'] else 'v1',
            'quantization': MODEL_CONFIG['qtype'] if MODEL_CONFIG['quantize'] else 'none',
            'low_vram': MODEL_CONFIG['low_vram'],
            'network_type': network_config['type'],
            'lora_rank': network_config['linear'],
            'lora_alpha': network_config['linear_alpha'],
            'learning_rate': train_config['learning_rate'],
            'total_steps': train_config['steps'],
            'batch_size': train_config['batch_size'],
            'optimizer': train_config['optimizer'],
            'num_images': num_images,
            'num_captions': num_captions,
            'seed': GLOBAL_SEED,
            'sample_every': SAMPLE_CONFIG['sample_every'],
            'experiment_name': variation['name'] if variation else None,
        }
        
        monitor = TrainingMonitor(
            output_dir='/content/output',
            config=config_dict,
            project=WANDB_PROJECT,
            entity=WANDB_ENTITY,
            run_name=run_name,
            group=group_name
        )
        
        # Log dataset
        monitor.log_dataset_info('/content/dataset')
        
        # Start monitoring
        monitor.start_monitoring()
    
    # Run training
    print("\n" + "="*60)
    print(f"STARTING TRAINING: {run_name}")
    print("="*60)
    
    start_time = time.time()
    
    try:
        run_job(job_config)
        
        training_time = time.time() - start_time
        print(f"\n✓ Training complete! Time: {training_time/60:.1f} min")
        
        if WANDB_ENABLED:
            # Log final model
            from pathlib import Path
            for model_file in Path('/content/output').glob('*.safetensors'):
                monitor.log_final_model(model_file)
            
            monitor.finish({'training_time_minutes': training_time/60})
        
        return run_name
        
    except Exception as e:
        print(f"\n❌ Training failed: {e}")
        if WANDB_ENABLED:
            monitor.finish({'error': str(e)})
        raise

# ========== RUN TRAINING ==========
if EXPERIMENT_MODE:
    print(f"\n🔬 Running {len(EXPERIMENT_VARIATIONS)} experiments...\n")
    for i, var in enumerate(EXPERIMENT_VARIATIONS, 1):
        print(f"\nEXPERIMENT {i}/{len(EXPERIMENT_VARIATIONS)}: {var['name']}")
        run_training_with_wandb(var)
else:
    run_training_with_wandb()

## 📊 Step 7: Generate X/Y Comparison Grids

In [ ]:
from xy_grid import XYGridGenerator
from pathlib import Path

# Find trained models
output_dir = Path('/content/output')
model_files = list(output_dir.glob('*.safetensors'))

if not model_files:
    print("⚠ No models found to compare")
else:
    print(f"Found {len(model_files)} models")
    
    # Create grid generator
    grid_gen = XYGridGenerator(device='cuda')
    
    # Test prompts
    test_prompts = [
        "a beautiful landscape with mountains",
        "portrait of a person",
        "a cute animal"
    ]
    
    # Compare models
    if len(model_files) > 1:
        grid_gen.compare_models(
            model_paths=[str(m) for m in model_files[:3]],
            prompts=test_prompts,
            output_path='/content/model_comparison.png',
            width=512,
            height=512,
            num_inference_steps=8,
            guidance_scale=0
        )
        print("✓ Model comparison saved")
    
    # Compare seeds
    grid_gen.compare_seeds(
        prompt=test_prompts[0],
        seeds=[42, 123, 456, 789],
        output_path='/content/seed_comparison.png',
        width=512,
        height=512
    )
    print("✓ Seed comparison saved")
    
    grid_gen.cleanup()

## ☁️ Step 8: Backup & Upload

In [ ]:
from upload_utils import UploadManager, quick_backup

# Quick backup to both Drive and HF
quick_backup(
    output_dir='/content/output',
    project_name=PROJECT_NAME
)

## 🎨 Step 9: Test Your LoRA

In [ ]:
from diffusers import ZImagePipeline
import torch
from pathlib import Path

# Load pipeline
pipe = ZImagePipeline.from_pretrained(
    "Tongyi-MAI/Z-Image-Turbo",
    torch_dtype=torch.bfloat16
).to("cuda")

# Find your LoRA
lora_files = list(Path('/content/output').glob('*.safetensors'))
if lora_files:
    lora_path = lora_files[-1]  # Use latest
    print(f"Loading LoRA: {lora_path.name}")
    pipe.load_lora_weights(str(lora_path))
    
    # Generate test image
    test_prompt = input("Enter your prompt: ")
    
    image = pipe(
        prompt=test_prompt,
        num_inference_steps=8,
        guidance_scale=0.0,
        width=1024,
        height=1024,
        generator=torch.Generator("cuda").manual_seed(42)
    ).images[0]
    
    # Display
    display(image)
    
    # Save
    image.save('/content/test_output.png')
    print("✓ Saved to /content/test_output.png")
else:
    print("⚠ No LoRA files found")

## 📝 Summary & Next Steps

**What we did:**
1. ✅ Trained Z-Image Turbo LoRA with W&B tracking
2. ✅ Logged all metrics, samples, and artifacts
3. ✅ Generated comparison grids
4. ✅ Backed up to Drive and HuggingFace

**Your outputs:**
- Models: `/content/output/*.safetensors`
- Samples: `/content/output/samples/`
- Grids: `/content/*_comparison.png`
- W&B Dashboard: Check your W&B project
- HuggingFace: https://huggingface.co/YOUR_USERNAME

**Next steps:**
1. Review metrics in W&B dashboard
2. Compare different checkpoints
3. Fine-tune hyperparameters if needed
4. Share your LoRA on HuggingFace!

---
Made with ❤️ for the AI community